# Bike Sharing Demand Prediction

**Name - Harikumar Thiruniraiselvam**</br>
**ID - 2025ac05989@wilp.bits-pilani.ac.in** </br>
**M.Tech AIML — Machine Learning Assignment 1**

## About the dataset

The data is about Bike Sharing Demand — an hourly log of bike rentals from a city bike share service.

There are three files given:

- **bike_train.csv** — the training set. Each row is one hour, with the conditions during that hour and the actual rental count. This is what the model learns from.
- **bike_test.csv** — the test set. Same conditions per hour, but the rental count is removed. We predict it.
- **SampleSubmission.csv** — shows the exact format the predictions need to be in.

For every hour the data records:

- **When it happened** — a datetime stamp.
- **Calendar context** — the season, whether the day was a holiday, and whether it was a working day.
- **Weather** — a coded weather situation (clear, misty, light rain/snow, heavy rain/snow), the actual temperature, the "feels like" temperature, the humidity and the wind speed.

The training file has three extra columns that the test file does not have: the number of **casual** riders (no membership), the number of **registered** riders (members), and the **total count** which is the sum of the two. The total count is our prediction target. The casual and registered breakdowns cannot be used as inputs because they are not available at prediction time.

Conceptually the problem is: given the time and the weather, how many bikes will get rented? The signal lives in two places — the calendar (people commute on weekday mornings and evenings, ride for leisure on weekend afternoons) and the weather (warm clear days drive demand up, cold or rainy days drive it down). Because the data spans two full years at hourly granularity, there is strong daily, weekly and seasonal structure on top of the weather effects.

**Goal:** Predict hourly bike rental counts from weather, time and seasonal features in the bike_test.csv.

### Importing necessary libraries

In [1]:
# Data Analysis and Visualization Libraries

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Machine Learning Libraries
from sklearn.feature_selection import mutual_info_regression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


### Load the dataset

In [2]:
# Training dataset

train = pd.read_csv('../data/bike_train.csv')
train.head()

,datetime,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,casual,registered,count
0,2012-07-15 7:00:00,3,0,0,1,28.70000,33.335000,79,6.003200,17,30,47
1,2012-08-14 15:00:00,3,0,1,1,33.62000,37.880000,46,15.001300,84,199,283
2,2011-02-06 6:00:00,1,0,0,1,10.66000,12.880000,60,15.001300,0,1,1
3,2012-05-06 17:00:02,2,0,0,2,26.42506,30.566166,61,9.512288,198,330,531
4,2012-01-09 2:00:00,1,0,1,1,9.84000,12.120000,56,8.998100,2,3,5


In [3]:
# Test dataset

test  = pd.read_csv('../data/bike_test.csv')
test.head()

,datetime,season,holiday,workingday,weather,temp,atemp,humidity,windspeed
0,05-06-2012 5:00,2,0,1,3,19.680000,23.48500,82,11.00140
1,19-03-2011 19:00,1,0,0,1,18.040000,21.97000,33,19.00120
2,02-04-2012 6:00,2,0,1,1,13.885038,16.24664,68,29.52646
3,13-07-2012 20:00,3,0,1,1,30.340000,33.33500,48,8.99810
4,09-11-2011 19:00,4,0,1,1,18.040000,21.97000,72,7.00150


**Observation**

- Train has 12 columns, test has 9 columns. `casual`, `registered` and `count` are only in train. `count` is the target. 
- Datetime format is different in the two files. Train is `YYYY-MM-DD HH:MM:SS`, test is `DD-MM-YYYY HH:MM`. Need to parse each one explicitly.
- Some train rows end in `:00:01` or `:00:02` instead of `:00:00`. On those rows `count` doesn't equal `casual + registered`. 

## Q1. Dataset size, missing values, feature types

In [4]:
print("=============================")
print("Training dataset information:")
print("=============================")
print("1. Rows/Datapoints : ", train.shape[0])
print("2. Columns/Features : ", train.shape[1])
print("3. Feature Types :")
print(train.dtypes)
print("4. Missing Values :")
print(train.isnull().sum())

Training dataset information:
1. Rows/Datapoints :  10450
2. Columns/Features :  12
3. Feature Types :
datetime       object
season          int64
holiday         int64
workingday      int64
weather         int64
temp          float64
atemp         float64
humidity        int64
windspeed     float64
casual          int64
registered      int64
count           int64
dtype: object
4. Missing Values :
datetime      0
season        0
holiday       0
workingday    0
weather       0
temp          0
atemp         0
humidity      0
windspeed     0
casual        0
registered    0
count         0
dtype: int64


In [5]:
print("=============================")
print("Test dataset information:")
print("=============================")
print("1. Rows/Datapoints : ", test.shape[0])
print("2. Columns/Features : ", test.shape[1])
print("3. Feature Types :")
print(test.dtypes)
print("4. Missing Values :")
print(test.isnull().sum())

Test dataset information:
1. Rows/Datapoints :  2613
2. Columns/Features :  9
3. Feature Types :
datetime       object
season          int64
holiday         int64
workingday      int64
weather         int64
temp          float64
atemp         float64
humidity        int64
windspeed     float64
dtype: object
4. Missing Values :
datetime      0
season        0
holiday       0
workingday    0
weather       0
temp          0
atemp         0
humidity      0
windspeed     0
dtype: int64


In [6]:
train.describe()

,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,casual,registered,count
count,10450.000000,10450.000000,10450.000000,10450.000000,10450.000000,10450.000000,10450.000000,10450.000000,10450.000000,10450.000000,10450.000000
mean,2.507943,0.028804,0.675694,1.413876,20.191700,23.605793,61.924211,12.765259,35.869091,154.511675,190.410526
std,1.116946,0.167263,0.468137,0.632258,7.792683,8.478045,19.245193,8.102821,49.629436,150.861267,181.038755
min,1.000000,0.000000,0.000000,1.000000,0.820000,0.760000,0.000000,0.000000,0.000000,0.000000,1.000000
25%,2.000000,0.000000,0.000000,1.000000,13.940000,16.665000,47.000000,7.001500,4.000000,35.000000,42.000000
50%,3.000000,0.000000,1.000000,1.000000,20.500000,24.240000,62.000000,12.450050,16.000000,117.000000,143.000000
75%,4.000000,0.000000,1.000000,2.000000,26.240000,31.060000,77.000000,16.997900,49.000000,221.000000,283.000000
max,4.000000,1.000000,1.000000,4.000000,41.000000,45.455000,100.000000,56.996900,367.000000,886.000000,977.000000


In [7]:
test.describe()

,season,holiday,workingday,weather,temp,atemp,humidity,windspeed
count,2613.000000,2613.000000,2613.000000,2613.000000,2613.000000,2613.000000,2613.000000,2613.000000
mean,2.509759,0.032912,0.683506,1.437046,20.247516,23.691936,61.980100,13.043217
std,1.113917,0.178441,0.465197,0.639108,7.812128,8.525166,19.252244,8.387861
min,1.000000,0.000000,0.000000,1.000000,1.466819,3.030000,0.000000,0.000000
25%,2.000000,0.000000,0.000000,1.000000,13.940000,16.665000,47.000000,7.001500
50%,3.000000,0.000000,1.000000,1.000000,20.057846,23.743546,62.000000,12.998000
75%,4.000000,0.000000,1.000000,2.000000,26.240000,31.060000,77.000000,17.634405
max,4.000000,1.000000,1.000000,3.000000,38.540000,44.555405,100.000000,56.996900


### Observation - Q1
- Train dataset has 10450 rows, test dataset has 2613 rows.
- No missing values in both training and test dataset, so there is no need of missing value imputation.
- There are 4 discrete variables `season`, `holiday`, `workingday`, `weather` represented in integer values, so we need to do one hot encoding in the feature engineering section.
- There are 4 continuous variables `temp`, `atemp`, `humidity`, `windspeed`.
- `casual` and `registered` exist only in train and sum to `count`. But we will consider only final `count`.

## Q2. Relationships between features and the target `count`

In [8]:
# Temporaryly fix the datetime feature in the training dataset for better visualization

train['datetime'] = pd.to_datetime(train['datetime'])
train['hour'] = train['datetime'].dt.hour
train['day'] = train['datetime'].dt.day
train['weekday'] = train['datetime'].dt.dayofweek
train['month'] = train['datetime'].dt.month
train['year'] = train['datetime'].dt.year

train.head()

,datetime,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,casual,registered,count,hour,day,weekday,month,year
0,2012-07-15 07:00:00,3,0,0,1,28.70000,33.335000,79,6.003200,17,30,47,7,15,6,7,2012
1,2012-08-14 15:00:00,3,0,1,1,33.62000,37.880000,46,15.001300,84,199,283,15,14,1,8,2012
2,2011-02-06 06:00:00,1,0,0,1,10.66000,12.880000,60,15.001300,0,1,1,6,6,6,2,2011
3,2012-05-06 17:00:02,2,0,0,2,26.42506,30.566166,61,9.512288,198,330,531,17,6,6,5,2012
4,2012-01-09 02:00:00,1,0,1,1,9.84000,12.120000,56,8.998100,2,3,5,2,9,0,1,2012


In [9]:
# Average rentals by hour of day 

hourly = train.groupby('hour', as_index=False)['count'].mean()
fig = px.line(hourly, x='hour', y='count', markers=True, title='Average rentals by hour of day')
fig.update_layout(xaxis=dict(tickmode='linear', dtick=1))
fig.show()

**Observation**

- Demand follows a clear daily rhythm: almost nothing overnight (lowest around 3-4am, ~7 rides), a sharp morning climb, and the busiest hours in the evening (5-6pm, ~430-468 rides).
- The curve is bimodal, so `hour` is one of the strongest predictors, but the relationship is clearly not linear.

In [10]:
# split by working day vs non-working day

hourly_wd = train.groupby(['hour', 'workingday'], as_index=False)['count'].mean()
hourly_wd['workingday'] = hourly_wd['workingday'].map({0: 'Non-working day', 1: 'Working day'})

fig = px.line(hourly_wd, x='hour', y='count', color='workingday', markers=True,
              title='Average rentals by hour of day, split by working day vs non-working day')
fig.update_layout(xaxis=dict(tickmode='linear', dtick=1), legend_title_text='')
fig.show()

**Observation**

- The two curves tell different stories. Working days have a commute double-peak at 8am and 5-6pm (peak ~522 at 17:00).
- Non-working days show a single broad afternoon hump around midday (peak ~396 at 14:00).
- So `hour` and `workingday` interact: what a given hour means for demand depends on whether it is a working day.

In [11]:
# numerical predictors vs count

num_cols = ['temp', 'atemp', 'humidity', 'windspeed']
sample = train[num_cols + ['count']].sample(2500, random_state=1)
melted = sample.melt(id_vars='count', var_name='feature', value_name='value')

fig = px.scatter(melted, x='value', y='count', facet_col='feature', facet_col_wrap=2,
                 opacity=0.4, title='count vs numerical features')
fig.update_xaxes(matches=None, showticklabels=True)
fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))
fig.show()

**Observation**

- `temp` and `atemp` push count the same way (both ~+0.39): warmer hours see more rides.
- `humidity` goes the other way (~-0.32): muggy hours are quieter.
- `windspeed` is nearly flat (~+0.10), so on its own it carries little signal.
- All four are noisy clouds rather than tight lines, so no single weather number explains count by itself.

In [12]:
# categorical features

cat_cols = ['season', 'weather', 'workingday']
melted = train[cat_cols + ['count']].melt(id_vars='count', var_name='feature', value_name='category')

fig = px.box(melted, x='category', y='count', facet_col='feature',
             title='count by season / weather / workingday')
fig.update_xaxes(matches=None, showticklabels=True)
fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))
fig.show()

**Observation**

- `season`: summer and fall (2 and 3) are clearly busier (mean ~213-234) than winter (1, ~117), with spring in between.
- `weather`: demand falls steadily as conditions worsen (clear ~204 down to light rain/snow ~122).
- `workingday`: the medians are close (124 vs 150), but the boxes are wide with many high outliers, which is the commute peaks showing through.

In [13]:
# Monthly demand trend, split by year

monthly = train.groupby(['year', 'month'], as_index=False)['count'].mean()
fig = px.line(monthly, x='month', y='count', color='year', markers=True,
              title='Average rentals by month (2011 vs 2012)')
fig.update_layout(xaxis=dict(tickmode='linear', dtick=1), legend_title_text='year')
fig.show()

**Observation**

- Demand roughly doubled from 2011 (mean ~144) to 2012 (mean ~237) - the service was clearly growing over time.
- Both years follow the same shape: low in winter, rising to a broad May-October peak. So `season`/`month` carry real signal.
- Since the two years sit at different levels, `year` is worth keeping as a feature, and any chart that blends both years lands somewhere in between.

In [14]:
# Average rentals by day of week

dow = train.groupby('weekday', as_index=False)['count'].mean()
labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
dow['weekday'] = dow['weekday'].map(dict(enumerate(labels)))
fig = px.bar(dow, x='weekday', y='count', title='Average rentals by day of week')
fig.show()

**Observation**

- The daily total is almost flat across the week (about 178-198), so the day on its own barely moves overall demand.
- The real day-of-week effect is in the *shape* over the hours (commute peaks vs a midday leisure hump), which the hour-by-working-day chart already showed.

In [15]:
# Casual vs registered riders by hour, split by working day

rider = (train.groupby(['workingday', 'hour'])[['casual', 'registered']]
              .mean().reset_index())
rider['workingday'] = rider['workingday'].map({0: 'Non-working day', 1: 'Working day'})
long = rider.melt(id_vars=['workingday', 'hour'], var_name='rider_type', value_name='avg')

fig = px.line(long, x='hour', y='avg', color='rider_type', facet_col='workingday',
              markers=True, title='Casual vs registered riders by hour')
fig.update_xaxes(matches=None, showticklabels=True, tickmode='linear', dtick=2)
fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))
fig.update_layout(legend_title_text='')
fig.show()

**Observation**

- Registered riders create the commute double-peak on working days (a peak near 466 at 17:00, another around 08:00) - this looks like people travelling to and from work.
- Casual riders barely register on working days but drive a single midday hump on non-working days (peak near 159 at 14:00) - more like weekend leisure trips.
- `count` is simply casual + registered, so these two groups explain the hour-by-working-day pattern. Note both columns are missing from the test set, so they help understanding only and cannot be used as model features.

In [16]:
# Demand vs weather, with how often each weather occurs

weather = train.groupby('weather', as_index=False).agg(
    avg_count=('count', 'mean'), rows=('count', 'size'))
fig = px.bar(weather, x='weather', y='avg_count', text='rows',
             title='Average rentals by weather (label = number of rows)')
fig.update_traces(texttemplate='n=%{text}', textposition='outside')
fig.update_layout(xaxis=dict(tickmode='linear', dtick=1))
fig.show()

**Observation**

- Demand drops steadily as weather worsens: clear/cloudy ~204, misty ~177, light rain/snow ~122.
- Weather category 4 (heavy rain/storm) appears in a single row, so it is effectively absent. Safest to fold it into category 3 so one-hot encoding doesn't create a column that shows up only once.

In [17]:
# correlation heatmap - numeric block only

corr_cols = ['season', 'holiday', 'workingday', 'weather', 'temp', 'atemp',
             'humidity', 'windspeed', 'hour', 'weekday', 'month', 'year', 'count']
corr = train[corr_cols].corr()

fig = px.imshow(corr, text_auto='.2f',
                color_continuous_scale='RdBu_r',
                color_continuous_midpoint=0,
                aspect='auto',
                title='Correlation heatmap')
fig.update_layout(width=780, height=620)
fig.show()

**Observation**

- `temp` and `atemp` are almost perfectly correlated (~0.98). They are the same signal, so we keep one and drop the other, or let regularization handle the collinearity.
- Looking at the `count` row, the strongest linear relationships are with `hour`, `temp`/`atemp` and `year` (rentals grew a lot from 2011 to 2012), and the strongest negative one is with `humidity`.
- `season` and `month` are highly correlated with each other (they encode the same time-of-year information at different granularities).
- `workingday` and `holiday` are negatively correlated (a holiday cannot be a working day). Which make sence.
- One caveat: this is Pearson correlation, which only measures the *linear* part. `hour` looks moderate here (~0.40), but its real effect is the non-linear double-peak seen in the hourly charts, so a small number on this map does not always mean a weak feature.
- `casual` and `registered` are left out on purpose. They add up to `count` (so they would leak the target) and they are not present in the test dataset.

## Q2 summary - what drives `count`

Pulling all the charts together:

**Time is the biggest driver.**
- `hour` matters most. Demand is near zero overnight and peaks in the evening, and the daily shape is a double-peak rather than a straight line. Plain correlation (0.40) understates it; measured as a non-linear effect it is the strongest feature.
- `workingday` reshapes that hourly curve: a commute double-peak (8am and 5-6pm) on working days versus a single afternoon hump on weekends and holidays. The two features interact.
- `year` captures growth: 2012 demand was roughly double 2011, so it is worth keeping as a feature.
- `season` and `month` give the yearly cycle (busy May-October, quiet winter). They are two views of the same thing.
- `weekday` on its own is almost flat; its effect lives in the hourly shape, not the daily total.

**Weather has a secondary but sensible effect.**
- Warmer temperature helps (`temp`/`atemp` ~+0.39), higher humidity hurts (~-0.32), and rides drop as the weather code worsens. `windspeed` barely moves the needle.

**Things to handle when modelling.**
- `temp` and `atemp` are ~0.98 correlated, and `season`/`month` overlap too, so expect some collinearity.
- `weather` category 4 has a single row; safest to fold it into category 3.
- `casual` and `registered` explain these patterns well but they sum to `count` and are missing from the test set, so they stay EDA-only and are not used as features.

Rough ranking of usable predictors by strength: hour (non-linear) > temp/atemp > humidity > year > season/month > weather > windspeed, with weekday and holiday weak on their own.

## Q3. Suggest which variables are likely to be most informative

In [18]:
# Rank features by how much they tell us about count

feats = ['hour', 'temp', 'atemp', 'humidity', 'year', 'season', 'month',
         'weather', 'windspeed', 'workingday', 'weekday', 'holiday']

discrete = [f in ['hour', 'year', 'season', 'month', 'weather',
                  'workingday', 'weekday', 'holiday'] for f in feats]

mi = mutual_info_regression(train[feats], train['count'], discrete_features=discrete, random_state=0)
mi = pd.Series(mi, index=feats).sort_values()

fig = px.bar(x=mi.values, y=mi.index, orientation='h',
             title='Feature importance (mutual information with count)')
fig.update_layout(xaxis_title='mutual information', yaxis_title='')
fig.show()

### Q3 - Most informative features

Using the EDA together with the mutual-information scores above (mutual information is used because it also picks up non-linear links, which plain correlation misses):

**Most informative**
- **`hour`** is by far the strongest signal (MI ~0.63). The catch is the relationship is a non-linear double-peak, so it should be encoded properly (one-hot per hour, or a cyclic / polynomial form) rather than dropped in as a single linear term.
- **`temp`** (with `atemp`) is the strongest continuous driver (MI ~0.15, correlation +0.40): warmer hours mean more rides. `temp` and `atemp` are ~0.98 correlated, so keeping one of them is enough.
- **`humidity`** has a clear negative effect (MI ~0.11, correlation -0.32): muggy hours are quieter.
- **`workingday`** looks weak on its own (MI ~0.02) but is genuinely useful because of how it interacts with `hour`. It flips the daily shape from a commute double-peak to a single weekend hump, so an hour-by-workingday interaction is worth far more than the raw column.
- **`year`** is a cheap, useful level feature: 2012 demand was about double 2011.
- **`season` / `month`** carry the yearly cycle (busy in summer and autumn). They overlap heavily, so one of them is enough.

**Weak / secondary**
- **`weather`** points the right way (demand falls as it worsens) but adds little on its own and overlaps with `humidity` and `season`; its category 4 also has only a single row.
- **`windspeed`, `weekday`, `holiday`** carry very little signal by themselves.

**Important Features:** 

`hour` (encoded for its non-linear shape), `temp`, `humidity`, `workingday` (as an interaction with `hour`), `year`, and one of `season` are the variables most likely to drive a good model. The rest are minor.

## Q4. Feature Engineering

In [19]:
# Q4 - Feature Engineering
# hour, weekday, month and year were already derived from datetime in the EDA step (Q2),
# so we reuse them. Following the Q3 feature-selection decision we keep ONLY the variables
# that looked informative - season, year, hour, workingday (as an hour interaction),
# temp and humidity - and drop the weak ones (weather, windspeed, weekday, holiday, month, atemp).

# train datetime was already parsed in EDA; the test file uses a different format, parse it too
if train['datetime'].dtype == object:
    train['datetime'] = pd.to_datetime(train['datetime'], format='%Y-%m-%d %H:%M:%S')
if test['datetime'].dtype == object:
    test['datetime'] = pd.to_datetime(test['datetime'], format='%d-%m-%Y %H:%M')

def add_time_features(df):
    df['hour'] = df['datetime'].dt.hour
    df['weekday'] = df['datetime'].dt.dayofweek
    df['month'] = df['datetime'].dt.month
    df['year'] = df['datetime'].dt.year
    return df

train = add_time_features(train)
test = add_time_features(test)

def build_features(df):
    # one-hot the selected categorical codes (season, year, hour)
    base = pd.get_dummies(df[['season', 'year', 'hour']].astype('category'),
                          columns=['season', 'year', 'hour'])
    # workingday enters as an interaction with hour (not a flag): one-hot hour is the
    # non-working-day shape, hour * workingday lets working days have their own hourly profile
    hour_oh = pd.get_dummies(df['hour'].astype('category'), prefix='hour')
    interaction = hour_oh.mul(df['workingday'].values, axis=0)
    interaction.columns = [c + '_x_workingday' for c in hour_oh.columns]
    return pd.concat([base, interaction,
                      df[['temp', 'humidity']].reset_index(drop=True)], axis=1)

X = build_features(train).astype(float)
X_test = build_features(test).reindex(columns=X.columns, fill_value=0).astype(float)

# target: model log1p(count) because count is right-skewed and RMSLE lives in log space
y = np.log1p(train['count'])

print('selected-feature train matrix:', X.shape)
print('selected-feature test matrix :', X_test.shape)
print('train/test columns match:', list(X.columns) == list(X_test.columns))
print('count skew %.2f  ->  log1p(count) skew %.2f' % (train['count'].skew(), y.skew()))

selected-feature train matrix: (10450, 56)
selected-feature test matrix : (2613, 56)
train/test columns match: True
count skew 1.26  ->  log1p(count) skew -0.90


### Q4 - Feature Engineering

Feature engineering is optional in the brief, but it helps the linear model, so we engineer the time information and then keep only the features chosen in Q3.

**Already derived during EDA (reused, not redone).** `hour`, `weekday`, `month` and `year` (and `day`) were pulled out of `datetime` back in Q2 for the charts, so we reuse them here. `season` is already a raw column.

**Selected features (from the Q3 decision).** We keep only the informative variables: `season`, `year`, `hour`, `workingday` (as an interaction with `hour`), `temp` and `humidity`. The weak ones from Q3 are dropped: `weather`, `windspeed`, `weekday`, `holiday`, the redundant `month` (overlaps with `season`) and `atemp` (~0.98 correlated with `temp`).

**Encoding and transformations:**
1. **Fixed the test datetime.** The test file stores dates as `dd-mm-yyyy hh:mm`, a different format from train, so it is parsed explicitly and given the same time features.
2. **One-hot encoded `season`, `year` and `hour`.** These integer codes are not really ordered, so one-hot gives each level its own coefficient. For `hour` this is what lets a linear model bend to the morning/evening double-peak instead of one straight-line slope.
3. **Used `workingday` as an interaction with `hour`, not as a flag.** EDA showed the hourly shape is completely different on working vs non-working days (commute double-peak vs a single midday hump). A plain flag only shifts the whole curve up or down, whereas the interaction (`hour` one-hot times `workingday`) lets the model learn a separate hourly profile for each day type. In a quick check this single change moved validation RMSLE the most. The standalone workingday flag is then redundant (it equals the sum of the interaction columns), so we leave it out.
4. **Kept `temp` and `humidity`** as the continuous block.

This gives a 56-column matrix for both train and test (`X`, `X_test`) with identical columns and no missing values.


## Outlier removal

Right after feature engineering we drop the extreme `count` rows from the training data using the IQR rule, so the split and models in Q5/Q6 are built on the cleaned data. The test matrix `X_test` is left untouched.

In [20]:
# Drop extreme count rows (IQR rule) from the training data, after feature engineering.
# Only X / y (training) are trimmed; X_test is left untouched so we can still predict every test row.
counts = train['count']
q1, q3 = counts.quantile(0.25), counts.quantile(0.75)
upper_fence = q3 + 1.5 * (q3 - q1)
inliers = (counts <= upper_fence).values

fig = px.box(x=counts, points='outliers',
             title='count in the training data (points past the whisker are outliers)')
fig.add_vline(x=upper_fence, line_dash='dash', line_color='red',
              annotation_text='IQR fence = %.0f' % upper_fence)
fig.update_layout(xaxis_title='count', height=320)
fig.show()

if len(X) == len(inliers):          # apply once - safe to re-run this cell
    X = X[inliers].reset_index(drop=True)
    y = y[inliers].reset_index(drop=True)
print('IQR fence = %.0f | removed %d outlier rows | %d training rows remain'
      % (upper_fence, (~inliers).sum(), len(X)))

IQR fence = 644 | removed 292 outlier rows | 10158 training rows remain


### Observation - outlier removal

- The IQR rule flags **292 rows (~2.8%)** with `count` above ~644 (the busiest rush hours, up to 977 rentals); we drop them from the training data and leave `X_test` untouched.
- Q5 now splits from this cleaned data, so the validation RMSLE in Q5/Q6 comes out a little lower than before. Part of that is a cleaner fit and part is simply that the validation set no longer holds the hardest extreme-demand rows.
- Because the target is modelled as `log1p(count)`, the long right tail was already compressed, so the effect is modest rather than dramatic.

## Q5. Split data into training and validation sets and build a simple Linear Regression model

In [21]:
# Split the data into training and validation sets (80/20)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
print('training set:  ', X_train.shape)
print('validation set:', X_val.shape)

training set:   (8126, 56)
validation set: (2032, 56)


In [22]:
# Build a simple Linear Regression model

# RMSLE - the grading metric, exactly as given in the assignment brief
def rmsle(y_true, y_pred):
    return np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2))

linreg = LinearRegression()
linreg.fit(X_train, y_train)

# the model is trained on log1p(count); bring predictions back to counts with expm1
train_pred = np.clip(np.expm1(linreg.predict(X_train)), 0, None)
val_pred = np.clip(np.expm1(linreg.predict(X_val)), 0, None)

print('train RMSLE: %.4f' % rmsle(np.expm1(y_train), train_pred))
print('val RMSLE:   %.4f' % rmsle(np.expm1(y_val), val_pred))

train RMSLE: 0.4588
val RMSLE:   0.4433


## Q6. Improve performance: polynomial features, Ridge and Lasso

In [23]:
# Q6 - extend the feature space with polynomial terms, then apply Ridge and Lasso.
# Polynomial features (and scaling) are applied only to the continuous block (temp, humidity);
# the one-hot / interaction columns pass straight through.
def make_prep(degree):
    return ColumnTransformer([
        ('cont', Pipeline([('poly', PolynomialFeatures(degree=degree, include_bias=False)),
                           ('scale', StandardScaler())]), ['temp', 'humidity']),
        ('keep', 'passthrough', [c for c in X.columns if c not in ['temp', 'humidity']])
    ])

# collect every model we try, so Q7 can summarise them in one table
results = []

def evaluate(name, estimator, degree, alpha):
    estimator.fit(X_train, y_train)
    train_pred = np.clip(np.expm1(estimator.predict(X_train)), 0, None)
    val_pred = np.clip(np.expm1(estimator.predict(X_val)), 0, None)
    results.append({'model': name, 'degree': degree, 'alpha': alpha,
                    'rmsle_train': rmsle(np.expm1(y_train), train_pred),
                    'rmsle_val': rmsle(np.expm1(y_val), val_pred),
                    'estimator': estimator})

# baseline from Q5: plain Linear Regression (no polynomial)
evaluate('Linear (no poly)', LinearRegression(), 1, None)

# polynomial features + Linear Regression (degree 2 and 3)
evaluate('Poly2 + Linear', Pipeline([('prep', make_prep(2)), ('m', LinearRegression())]), 2, None)
evaluate('Poly3 + Linear', Pipeline([('prep', make_prep(3)), ('m', LinearRegression())]), 3, None)

# Ridge on degree-2 features, tuning the regularisation strength alpha
for a in [0.01, 0.1, 1.0, 10.0, 100.0]:
    evaluate('Poly2 + Ridge', Pipeline([('prep', make_prep(2)), ('m', Ridge(alpha=a))]), 2, a)

# Lasso on degree-2 features, tuning alpha
for a in [0.0001, 0.001, 0.01, 0.1]:
    evaluate('Poly2 + Lasso', Pipeline([('prep', make_prep(2)), ('m', Lasso(alpha=a, max_iter=10000))]), 2, a)

results_df = pd.DataFrame(results)[['model', 'degree', 'alpha', 'rmsle_train', 'rmsle_val']]
print(results_df.to_string(index=False))

best_i = results_df['rmsle_val'].idxmin()
best_model = results[best_i]['estimator']
best_name = results_df.loc[best_i, 'model']
print('\nbest model: %s (degree=%s, alpha=%s)  ->  val RMSLE %.4f'
      % (best_name, results_df.loc[best_i, 'degree'],
         results_df.loc[best_i, 'alpha'], results_df.loc[best_i, 'rmsle_val']))

           model  degree    alpha  rmsle_train  rmsle_val
Linear (no poly)       1      NaN     0.458825   0.443312
  Poly2 + Linear       2      NaN     0.443277   0.428530
  Poly3 + Linear       3      NaN     0.440921   0.426267
   Poly2 + Ridge       2   0.0100     0.443277   0.428532
   Poly2 + Ridge       2   0.1000     0.443278   0.428553
   Poly2 + Ridge       2   1.0000     0.443371   0.428840
   Poly2 + Ridge       2  10.0000     0.450079   0.437222
   Poly2 + Ridge       2 100.0000     0.566716   0.556944
   Poly2 + Lasso       2   0.0001     0.443340   0.428730
   Poly2 + Lasso       2   0.0010     0.449458   0.436083
   Poly2 + Lasso       2   0.0100     0.603397   0.596628
   Poly2 + Lasso       2   0.1000     1.263926   1.243493

best model: Poly3 + Linear (degree=3, alpha=nan)  ->  val RMSLE 0.4263


In [24]:
# Chart of the Q6 results table: validation RMSLE for every model tried
chart_df = results_df.copy()
chart_df['label'] = chart_df.apply(
    lambda r: r['model'] if pd.isna(r['alpha']) else '%s (a=%s)' % (r['model'], r['alpha']), axis=1)
order = chart_df.sort_values('rmsle_val', ascending=False)['label'].tolist()

fig = px.bar(chart_df, x='rmsle_val', y='label', orientation='h', color='model',
             category_orders={'label': order}, text='rmsle_val',
             title='Q6 model comparison - validation RMSLE (lower is better)',
             labels={'rmsle_val': 'validation RMSLE', 'label': '', 'model': 'model family'})
fig.update_traces(texttemplate='%{text:.4f}', textposition='outside')
fig.update_xaxes(range=[0, chart_df['rmsle_val'].max() * 1.12])
fig.update_layout(height=470)
fig.show()

In [25]:
# alpha vs validation RMSLE for Ridge and Lasso (alpha on a log scale)
fig = make_subplots(rows=1, cols=2, subplot_titles=('Ridge', 'Lasso'))
for col, fam in [(1, 'Poly2 + Ridge'), (2, 'Poly2 + Lasso')]:
    d = results_df[results_df['model'] == fam].sort_values('alpha')
    fig.add_trace(go.Scatter(x=d['alpha'], y=d['rmsle_val'], mode='lines+markers'), row=1, col=col)
    fig.update_xaxes(type='log', title_text='alpha', row=1, col=col)
fig.update_yaxes(title_text='validation RMSLE', row=1, col=1)
fig.update_layout(title='Regularization strength vs validation RMSLE', showlegend=False, height=380)
fig.show()

In [26]:
# Chart: train vs validation RMSLE per model - shows the overfitting / generalisation gap
tradeoff = results_df.copy()
tradeoff['label'] = tradeoff.apply(
    lambda r: r['model'] if pd.isna(r['alpha']) else '%s (a=%s)' % (r['model'], r['alpha']), axis=1)
order = tradeoff.sort_values('rmsle_val', ascending=False)['label'].tolist()

long = tradeoff.melt(id_vars='label', value_vars=['rmsle_train', 'rmsle_val'],
                     var_name='split', value_name='rmsle')
long['split'] = long['split'].map({'rmsle_train': 'train', 'rmsle_val': 'validation'})

fig = px.bar(long, x='rmsle', y='label', color='split', orientation='h', barmode='group',
             category_orders={'label': order, 'split': ['train', 'validation']},
             title='Train vs validation RMSLE per model (gap = overfitting)',
             labels={'rmsle': 'RMSLE', 'label': '', 'split': ''})
fig.update_layout(height=560, legend_title_text='')
fig.show()

### Q6 - observation

- Adding **degree-2** polynomial terms on `temp` and `humidity` drops validation RMSLE from about **0.443** (plain linear) to **0.429**, because demand can now curve with temperature instead of following a straight line.
- **Degree 3** edges it out a little further (val **0.426**); on this outlier-trimmed data its train error also drops, so there is no real overfitting and it comes out as the best model.
- **Ridge** and **Lasso** barely change things: the best Ridge sits at a small `alpha = 0.01` (val 0.429) and the best Lasso at `alpha = 0.0001` (val 0.429), both essentially matching plain poly + linear. Large `alpha` shrinks useful terms and the error climbs sharply.
- Train and validation RMSLE stay close throughout, so there is little overfitting for regularisation to remove - the **polynomial degree** is the main lever here.
- **Best model:** Poly3 + Linear, validation RMSLE **0.426**. Every run is stored in `results_df` for the Q7 summary.

## Q7. Summary of all models tried (RMSLE)

In [29]:
# Q7 - summarise every model family at its best setting, in one table

summary = results_df.sort_values('rmsle_val').groupby('model', as_index=False).first()
summary['note'] = summary['model']
summary = (summary.sort_values('rmsle_val')
                  .round({'rmsle_train': 4, 'rmsle_val': 4})
                  .reset_index(drop=True))
summary[['model', 'degree', 'alpha', 'rmsle_train', 'rmsle_val', 'note']]

,model,degree,alpha,rmsle_train,rmsle_val,note
0,Poly3 + Linear,3,NaN,0.4409,0.4263,Poly3 + Linear
1,Poly2 + Linear,2,NaN,0.4433,0.4285,Poly2 + Linear
2,Poly2 + Ridge,2,0.0100,0.4433,0.4285,Poly2 + Ridge
3,Poly2 + Lasso,2,0.0001,0.4433,0.4287,Poly2 + Lasso
4,Linear (no poly),1,NaN,0.4588,0.4433,Linear (no poly)


### Q7 - key observations

- Every model lands around **0.43** validation RMSLE except the plain linear baseline (**0.44**). The one real jump comes from adding **polynomial temperature/humidity terms**, not from anything else.
- The best model is **Poly3 + Linear (0.426)**, narrowly ahead of Poly2 + Linear and the regularised models (all ~0.429).
- **Ridge and Lasso do not beat plain poly + linear.** Train and validation RMSLE are almost equal, so there is little overfitting for regularisation to fix; strong `alpha` only makes things worse.
- The winning recipe is: rich time encoding (one-hot `hour` + `hour x workingday`) + a polynomial weather curve + a `log1p(count)` target.

## Q8. Residual plot for the best model

In [ ]:
# Q8 - residuals of the best model on the validation set

count_val = np.expm1(y_val)
pred_val = np.clip(np.expm1(best_model.predict(X_val)), 0, None)
residuals = count_val.values - pred_val

fig = make_subplots(rows=1, cols=2, subplot_titles=('Residuals vs predicted', 'Residual distribution'))
fig.add_trace(go.Scatter(x=pred_val, y=residuals, mode='markers', marker=dict(opacity=0.4)), row=1, col=1)
fig.add_hline(y=0, line_dash='dash', line_color='red', row=1, col=1)
fig.add_trace(go.Histogram(x=residuals, nbinsx=50), row=1, col=2)
fig.update_xaxes(title_text='predicted count', row=1, col=1)
fig.update_yaxes(title_text='residual (actual - predicted)', row=1, col=1)
fig.update_xaxes(title_text='residual', row=1, col=2)
fig.update_layout(title='Residuals for the best model: %s' % best_name, showlegend=False, height=400)
fig.show()

print('mean residual: %.1f bikes (slight under-prediction)  |  std: %.1f' % (residuals.mean(), residuals.std()))

mean residual: 7.8 bikes (slight under-prediction)  |  std: 55.4


### Q8 - observation

- The residuals sit close to zero and are roughly symmetric, so the model is not systematically biased - it gets the overall demand level right.
- Their spread **widens as the predicted count grows** (a funnel shape): the model is off by a few bikes at quiet hours and by tens of bikes at busy hours. That is expected for count data, and it is exactly why we score with RMSLE (relative error) rather than RMSE.
- In log space, where the model is actually fit, the mean residual is essentially zero (-0.007). The small positive mean in count space (~8 bikes) is the usual mild under-prediction that comes from training on `log1p(count)`.
- There is no leftover curved pattern in the residuals, which says the polynomial terms captured the temperature/humidity relationship well.

## Q9. Why the winning model performs better

### Q9 - explanation

The winning model is **Poly3 + Linear** (validation RMSLE 0.426). It beats the alternatives for a few connected reasons.

1. **It captures the non-linear weather effect.** A plain linear model forces demand to move in a straight line with `temp` and `humidity`. In reality demand rises with warmth and then flattens when it gets too hot or muggy - a curve. The polynomial terms let the model bend to that curve, which is where almost all of the gain over the baseline comes from (0.44 -> 0.43).

2. **The time pattern was already handled in feature engineering.** The one-hot `hour` and the `hour x workingday` interaction let even a linear model follow the commute double-peak on working days and the midday weekend hump. So the polynomial only had to add the weather curvature, not learn the time-of-day shape.

3. **Degree 3 helps once outliers are removed.** With the extreme rush-hour rows trimmed, the extra degree-3 flexibility no longer overfits - both train and validation error drop - so it edges out degree 2 by a small margin.

4. **Regularisation could not improve it.** Ridge and Lasso only help when a model overfits (validation much worse than train). Here train and validation RMSLE are almost equal, so there is nothing to shrink away; regularisation at best matched the plain fit and, at strong `alpha`, clearly hurt by dropping useful terms.

5. **Training on `log1p(count)` matches the metric.** RMSLE is squared error in log space, so fitting in log space optimises exactly what the model is graded on and stops the large rush-hour counts from dominating.

In short, the winner pairs an honest, flexible feature set (rich time encoding + a polynomial weather curve) with a target transform that matches RMSLE. It wins by modelling curvature the linear baseline cannot - not by being heavily regularised.

## Q10. Why does RMSLE penalize under-predictions more gently than RMSE?

In [30]:
# Q10 - how RMSE and RMSLE penalise an under-prediction as it worsens (actual = 1000)
actual = 1000
demo = pd.DataFrame({'prediction': [1000, 800, 500, 200, 0]})
demo['abs_error'] = actual - demo['prediction']
demo['RMSE_penalty (squared error)'] = demo['abs_error'] ** 2
demo['log_error'] = np.log1p(actual) - np.log1p(demo['prediction'])
demo['RMSLE_penalty (squared log)'] = demo['log_error'] ** 2
demo.round(3)

,prediction,abs_error,RMSE_penalty (squared error),log_error,RMSLE_penalty (squared log)
0,1000,0,0,0.000,0.000
1,800,200,40000,0.223,0.050
2,500,500,250000,0.692,0.479
3,200,800,640000,1.605,2.577
4,0,1000,1000000,6.909,47.731


### Q10 - answer

RMSE works on the **absolute** error `(actual - pred)`, squared. RMSLE works on the **log / ratio** error `log(1+actual) - log(1+pred)`, squared. The table above (for an actual of 1000) shows what that does to an under-prediction:

- RMSE's penalty is the squared absolute miss, so it grows **quadratically** - from 0 up to 1,000,000 as the prediction falls to 0.
- RMSLE's penalty is the squared log-ratio, so it grows only **logarithmically** - up to about 48 - and it is **bounded**: the worst case (predicting 0) costs just `log(1+actual)²`.

Because RMSLE measures the *proportional* shortfall rather than the raw gap, a large absolute under-prediction on a busy, high-count hour barely moves RMSLE but blows up RMSE. That is the sense in which RMSLE is gentler on under-predictions, and it is why we train on `log1p(count)` - the model is optimised for relative error, matching the metric, instead of being dominated by the biggest counts.

Moreover RMSLE the two directions are not symmetric - it penalises an under-prediction slightly more than an over-prediction of the same size - but both stay far below RMSE's absolute-error penalty on a large miss.

## Q11. What are the trade-offs between model simplicity and predictive power?

### Q11 - answer

Our own results (Q7) show the trade-off directly.

- **Too simple underfits.** The plain linear model (no polynomial) is the most interpretable but has the worst error (RMSLE **0.443**) - it cannot bend with temperature. Forcing simplicity through heavy regularisation does the same thing: Ridge at `alpha = 100` shrinks the coefficients toward a simpler model and the error jumps to **~0.56**.
- **More complexity buys accuracy - up to a point.** Adding degree-2 polynomial terms on the weather variables captures the temperature curve and drops RMSLE to **0.429**. That is the one large, worthwhile gain.
- **Diminishing returns and overfitting.** Degree 3 improves validation only from 0.429 to **0.426** - a tiny gain for more parameters - and before outlier removal degree 3 actually overfit (training error kept falling while validation did not). This is the **bias-variance trade-off**: more flexibility lowers bias but raises variance and the risk of fitting noise.
- **Hidden costs of complexity.** More terms are harder to interpret and explain, slower to fit, and more sensitive to data quirks; our final design is already 50+ columns.

So the sweet spot is a **moderately** complex model - rich time features plus a low-degree weather polynomial - that captures the real structure without overfitting or becoming unexplainable. Past that point predictive power barely moves while simplicity, robustness and interpretability are lost.

## Q12. Why can't Linear Regression alone capture time-of-day effects effectively?

In [31]:
# Q12 - linear regression with hour as a single RAW number vs hour ONE-HOT (same other features)
fence = train['count'].quantile(0.75) + 1.5 * (train['count'].quantile(0.75) - train['count'].quantile(0.25))
keep = (train['count'] <= fence).values   # same outlier mask used after feature engineering

common = pd.get_dummies(train[['season', 'year']].astype('category'))
hour_oh = pd.get_dummies(train['hour'].astype('category'), prefix='hour')
inter = hour_oh.mul(train['workingday'].values, axis=0)
inter.columns = [c + '_x_workingday' for c in hour_oh.columns]

raw_hour    = pd.concat([common, train[['temp', 'humidity', 'hour', 'workingday']]], axis=1).astype(float)[keep]
onehot_hour = pd.concat([common, hour_oh, inter, train[['temp', 'humidity']]], axis=1).astype(float)[keep]
target = np.log1p(train['count'])[keep]

for label, M in [('hour as a raw number', raw_hour), ('hour one-hot + interaction', onehot_hour)]:
    Xt, Xv, yt, yv = train_test_split(M, target, test_size=0.2, random_state=42)
    pred = np.clip(np.expm1(LinearRegression().fit(Xt, yt).predict(Xv)), 0, None)
    print('%-30s val RMSLE %.4f' % (label, rmsle(np.expm1(yv), pred)))

hour as a raw number           val RMSLE 1.0086
hour one-hot + interaction     val RMSLE 0.4433


### Q12 - answer

Time-of-day demand is **non-linear and non-monotonic**. The Q2 EDA showed `count` versus `hour` is a **double-peak** curve - almost nothing overnight, a sharp peak near 8am, a midday dip, and the day's high around 5-6pm.

If `hour` is given to linear regression as one numeric column (0-23), the model can only learn a **single coefficient** for it - effectively a straight line `count ~ b*hour`. A straight line can only slope up or down across the day; it can never produce two separate peaks with a dip between them, and it ignores that hour is **cyclical** (23 sits next to 0). The cost is visible in the demo above: with `hour` as a raw number the validation RMSLE is **~1.01**, more than double the **~0.44** we get once `hour` is one-hot encoded. Q3 told the same story - the *linear* correlation of `hour` with `count` is only ~0.40, while its true non-linear association (correlation ratio) is ~0.72, so a single linear term can use only a small part of the signal.

The fix is feature engineering, which is what Q4 does: **one-hot encoding** `hour` gives every hour its own coefficient, so a linear model can trace any shape across the day, and the `hour x workingday` interaction lets working days and weekends differ. So the limitation is not linear regression itself - it is using `hour` as a raw linear input. Supply the non-linearity through features (one-hot, interactions, splines, or sine/cosine encodings) and a linear model handles time-of-day fine.

## Submission file

We refit the best model (from Q7) on **all** the cleaned training data and predict `bike_test.csv`. Predictions are made in log space and turned back to counts with `expm1`, then clipped at 0. The output follows the `SampleSubmission.csv` format (`datetime, Count_Predicted`).

In [33]:
# Refit the best model on all cleaned training data, then predict the test set
best_model.fit(X, y)
test_pred = np.clip(np.expm1(best_model.predict(X_test)), 0, None)

# count is always a whole number, so round the predictions to integers (submission only)
test_pred = np.round(test_pred).astype(int)

# bike_test.csv already stores datetime in the SampleSubmission format and order, so reuse it directly
submission = pd.read_csv('../data/bike_test.csv', usecols=['datetime'])
submission['Count_Predicted'] = test_pred
submission.to_csv('submission.csv', index=False)

print('best model refit on full cleaned data:', best_name)
print('submission saved -> submission.csv  (%d rows, columns %s)' % (len(submission), list(submission.columns)))
print('prediction range: %d to %d  (mean %.1f)' % (test_pred.min(), test_pred.max(), test_pred.mean()))
submission.head()

best model refit on full cleaned data: Poly3 + Linear
submission saved -> submission.csv  (2613 rows, columns ['datetime', 'Count_Predicted'])
prediction range: 1 to 1012  (mean 187.3)


,datetime,Count_Predicted
0,05-06-2012 5:00,29
1,19-03-2011 19:00,119
2,02-04-2012 6:00,116
3,13-07-2012 20:00,430
4,09-11-2011 19:00,269


### Observation - submission

- The best model **Poly3 + Linear** is refit on the full cleaned training set (not just the 80% split) so the final predictions use all available data.
- Since `count` is always a whole number in the data, the final predictions are **rounded to the nearest integer** for the submission. This rounding is applied only here - the RMSLE evaluations in Q5-Q8 still use the raw real-valued predictions.
- `submission.csv` has **2613 rows** with columns `datetime, Count_Predicted`, matching `SampleSubmission.csv`, with whole-number predictions and no missing values.
- The predicted counts span roughly **1 to ~1000 with a mean near 187**, in line with the training data (mean ~190) - a good sanity check that the `log1p` -> `expm1` round-trip produced realistic demand values.